# Notebook 02: Làm sạch dữ liệu, Kỹ thuật đặc trưng & Phân nhóm dấu vân tay (Preprocessing & Feature Engineering)

**Đề tài:** Dự đoán hủy đặt phòng khách sạn tại thời điểm xác nhận đặt phòng (Booking Confirmation)  
**Đầu vào:** `raw/hotel_bookings.csv`  
**Đầu ra duy nhất:** `processed/hotel_clean.csv` (chứa `METADATA_COLS`, `MODEL_FEATURES`, và biến mục tiêu `is_canceled`).

### Nguyên tắc phương pháp luận nghiêm ngặt
1. **Không loại bỏ dòng trùng lặp một cách mù quáng**: Giữ nguyên toàn bộ bản ghi hợp lệ để bảo toàn tỷ lệ hủy thực tế (~37%) và quy mô đặt phòng.
2. **Loại bỏ rò rỉ dữ liệu & tuân thủ cutoff xác nhận đặt phòng**:
   - Loại bỏ hoàn toàn: `reservation_status`, `reservation_status_date`, `assigned_room_type`, `booking_changes`.
   - Loại khỏi `MODEL_FEATURES`: `adr` và mọi biến dẫn xuất từ ADR (`adr_per_person`, `estimated_booking_value_proxy`). Chỉ giữ `adr` và `estimated_booking_value_proxy` trong `METADATA_COLS` để phục vụ mô phỏng kịch bản kinh doanh ở Notebook 04.
3. **Ngữ nghĩa giá trị thiếu minh bạch**:
   - `agent` thiếu → `'Unknown_Agent'` (không giả định là không có đại lý).
   - `company` → biến nhị phân `company_id_present` (không khẳng định thiếu mã = không có công ty liên quan), sau đó bỏ cột `company` thô (>94% thiếu).
   - `children` thiếu (4 dòng, 0.003%) → điền `0` (giả định được ghi nhận tường minh).
4. **Giữ Metadata phục vụ truy vết & phân tích thời gian (`METADATA_COLS`)**: `row_id`, `group_id`, `arrival_date`, `booking_date` (ước lượng = `arrival_date - lead_time`), `arrival_date_year`, `adr`, `estimated_booking_value_proxy`, `is_extreme_adr`. Các cột này **tuyệt đối không đưa vào `MODEL_FEATURES`**.
5. **Sinh `group_id` SAU Feature Engineering trên toàn bộ `MODEL_FEATURES`**: Đảm bảo mọi cặp bản ghi mà mô hình nhìn thấy giống hệt nhau đều thuộc cùng một nhóm, kiểm tra không có va chạm (zero collision) và báo cáo số nhóm có nhãn hỗn hợp.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

RAW_PATH = Path('raw/hotel_bookings.csv')
PROC_DIR = Path('processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = PROC_DIR / 'hotel_clean.csv'

df_raw = pd.read_csv(RAW_PATH)
print(f'Dữ liệu gốc: {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột')

Dữ liệu gốc: 119,390 dòng x 32 cột


## 1. Lọc bản ghi bất hợp lý vật lý & Kiểm tra ngoại lai cấp bản ghi

- Chỉ loại bỏ các bản ghi **chắc chắn bất hợp lý vật lý**:
  1. `adults + children + babies == 0` (đơn đặt phòng không có khách nào).
  2. `adr < 0` (đơn giá âm, lỗi điều chỉnh kế toán).
- **Không tự động xóa `adr > 1000`**: Gắn cờ `is_extreme_adr` trong `METADATA_COLS` để so sánh độ nhạy trong mô phỏng kinh doanh ở Notebook 04 (và lưu ý `adr` không nằm trong `MODEL_FEATURES`).

In [2]:
df = df_raw.copy()
df['row_id'] = np.arange(len(df), dtype=np.int64)

# Kiểm tra các điều kiện lọc vật lý
children_filled = df['children'].fillna(0)
zero_guests_mask = (df['adults'] + children_filled + df['babies']) == 0
neg_adr_mask = df['adr'] < 0
extreme_adr_mask = df['adr'] > 1000

print(f'Số dòng tổng khách = 0 (sẽ loại): {zero_guests_mask.sum():,}')
print(f'Số dòng ADR < 0 (sẽ loại):       {neg_adr_mask.sum():,}')
print(f'Số dòng ADR > 1000 (GIỮ LẠI & gắn cờ is_extreme_adr): {extreme_adr_mask.sum():,}')
if extreme_adr_mask.sum() > 0:
    display(df.loc[extreme_adr_mask, ['row_id', 'hotel', 'is_canceled', 'arrival_date_year', 'arrival_date_month',
                                      'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'adr', 'market_segment']])

# Áp dụng lọc vật lý
valid_mask = (~zero_guests_mask) & (~neg_adr_mask)
df = df.loc[valid_mask].reset_index(drop=True)
print(f'\nKích thước sau lọc vật lý: {df.shape[0]:,} dòng (đã loại {(~valid_mask).sum():,} dòng, giữ {(len(df)/len(df_raw))*100:.2f}%)')
print(f'Tỷ lệ hủy phòng sau lọc:   {df["is_canceled"].mean()*100:.2f}%')

Số dòng tổng khách = 0 (sẽ loại): 180
Số dòng ADR < 0 (sẽ loại):       1
Số dòng ADR > 1000 (GIỮ LẠI & gắn cờ is_extreme_adr): 1

Kích thước sau lọc vật lý: 119,209 dòng (đã loại 181 dòng, giữ 99.85%)
Tỷ lệ hủy phòng sau lọc:   37.08%


,row_id,hotel,is_canceled,arrival_date_year,arrival_date_month,stays_in_weekend_nights,stays_in_week_nights,adults,adr,market_segment
48515,48515,City Hotel,1,2016,March,0,1,2,5400.0,Offline TA/TO


## 2. Tạo cột Metadata, Chuẩn hóa giá trị thiếu & Loại bỏ cột rò rỉ

- Tạo `arrival_date` và ước lượng `booking_date = arrival_date - lead_time` (ghi chú: `booking_date` là ước lượng theo ngày lưu trú dự kiến trừ `lead_time`).
- Loại bỏ hoàn toàn: `reservation_status`, `reservation_status_date`, `assigned_room_type`, `booking_changes`.
- Chuẩn hóa giá trị thiếu theo đúng ngữ nghĩa:
  - `children`: điền `0` (chỉ có 4 dòng thiếu trong toàn bộ dữ liệu gốc — giả định hợp lý rằng đơn không ghi nhận trẻ em).
  - `country`: điền `'Unknown'`.
  - `agent`: chuẩn hóa về chuỗi ID, thiếu → `'Unknown_Agent'`.
  - `company`: tạo cờ `company_id_present`, bỏ cột `company` thô.
  - `meal`: gộp `'Undefined'` → `'SC'` (Self Catering / không kèm bữa ăn theo tài liệu gốc Antonio et al., 2019).

In [3]:
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df['arrival_month_num'] = df['arrival_date_month'].map(month_map).astype(np.int64)

# Tạo metadata thời gian
arrival_dt = pd.to_datetime(
    dict(year=df['arrival_date_year'], month=df['arrival_month_num'], day=df['arrival_date_day_of_month']),
    errors='raise'
)
df['arrival_date'] = arrival_dt.dt.strftime('%Y-%m-%d')
df['booking_date'] = (arrival_dt - pd.to_timedelta(df['lead_time'], unit='D')).dt.strftime('%Y-%m-%d')

# Gắn cờ extreme ADR & tính booking value proxy (CHỈ để ở METADATA_COLS, KHÔNG đưa vào MODEL_FEATURES)
df['is_extreme_adr'] = (df['adr'] > 1000).astype(np.int8)
total_nights_tmp = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['estimated_booking_value_proxy'] = (df['adr'] * total_nights_tmp).round(2)

# Xử lý thiếu theo ngữ nghĩa chuẩn
n_children_na = int(df['children'].isna().sum())
df['children'] = df['children'].fillna(0).astype(np.int64)
print(f'children thiếu đã điền 0: {n_children_na} dòng')

df['country'] = df['country'].fillna('Unknown').astype(str).str.strip()
df['agent_id_present'] = df['agent'].notna().astype(np.int8)
df['agent'] = df['agent'].apply(lambda x: f'Agent_{int(x)}' if pd.notna(x) else 'Unknown_Agent')

df['company_id_present'] = df['company'].notna().astype(np.int8)
df['meal'] = df['meal'].replace({'Undefined': 'SC'}).astype(str).str.strip()

# Loại bỏ các cột rò rỉ & cột company thô
DROP_COLS = ['reservation_status', 'reservation_status_date', 'assigned_room_type', 'booking_changes', 'company']
df = df.drop(columns=DROP_COLS)
print(f'Đã loại bỏ các cột: {DROP_COLS}')

children thiếu đã điền 0: 4 dòng
Đã loại bỏ các cột: ['reservation_status', 'reservation_status_date', 'assigned_room_type', 'booking_changes', 'company']


## 3. Kỹ thuật đặc trưng tiền lưu trú (Strict Booking-Confirmation Feature Engineering)

Toàn bộ các đặc trưng được tạo dưới đây **không sử dụng `adr`**, **không sử dụng `booking_changes`**, và **không sử dụng `assigned_room_type`**:
- **Lưu trú**: `total_nights`, `weekend_ratio`, `is_zero_night`.
- **Thành phần khách**: `total_guests`, `has_children_or_babies`, `is_single_adult`.
- **Tính mùa vụ**: `arrival_month_num`, `month_sin`, `month_cos`, `is_peak_season` (tháng 6–8).
- **Lịch sử & tín hiệu cam kết**: `has_previous_cancellations`, `prev_cancel_ratio`, `has_special_requests`, `has_parking`, `agent_id_present`, `company_id_present`, `is_portugal` (`country == 'PRT'`).

In [4]:
# 3.1 Đặc trưng lưu trú
df['total_nights'] = (df['stays_in_weekend_nights'] + df['stays_in_week_nights']).astype(np.int64)
df['weekend_ratio'] = np.where(df['total_nights'] > 0, df['stays_in_weekend_nights'] / df['total_nights'], 0.0).round(6)
df['is_zero_night'] = (df['total_nights'] == 0).astype(np.int8)

# 3.2 Đặc trưng khách
df['total_guests'] = (df['adults'] + df['children'] + df['babies']).astype(np.int64)
df['has_children_or_babies'] = ((df['children'] + df['babies']) > 0).astype(np.int8)
df['is_single_adult'] = ((df['adults'] == 1) & (df['children'] == 0) & (df['babies'] == 0)).astype(np.int8)

# 3.3 Đặc trưng mùa vụ (cyclical)
df['month_sin'] = np.sin(2 * np.pi * df['arrival_month_num'] / 12.0).round(6)
df['month_cos'] = np.cos(2 * np.pi * df['arrival_month_num'] / 12.0).round(6)
df['is_peak_season'] = df['arrival_month_num'].isin([6, 7, 8]).astype(np.int8)

# 3.4 Đặc trưng lịch sử & cam kết
prev_total = df['previous_cancellations'] + df['previous_bookings_not_canceled']
df['has_previous_cancellations'] = (df['previous_cancellations'] > 0).astype(np.int8)
df['prev_cancel_ratio'] = np.where(prev_total > 0, df['previous_cancellations'] / prev_total, 0.0).round(6)
df['has_special_requests'] = (df['total_of_special_requests'] > 0).astype(np.int8)
df['has_parking'] = (df['required_car_parking_spaces'] > 0).astype(np.int8)
df['is_portugal'] = (df['country'] == 'PRT').astype(np.int8)

# Chuẩn hóa tất cả cột chuỗi
TARGET_COL = 'is_canceled'
METADATA_COLS = [
    'row_id', 'group_id', 'arrival_date', 'booking_date', 'arrival_date_year',
    'adr', 'estimated_booking_value_proxy', 'is_extreme_adr'
]

# Danh sách MODEL_FEATURES chính thức (tuyệt đối không có metadata, target, adr, booking_changes, assigned_room_type)
MODEL_FEATURES = [c for c in df.columns if c not in METADATA_COLS and c != TARGET_COL]

# Kiểm tra chắc chắn không có cột bị cấm nào lọt vào MODEL_FEATURES
FORBIDDEN_IN_MODEL = {
    'is_canceled', 'row_id', 'group_id', 'arrival_date', 'booking_date', 'arrival_date_year',
    'reservation_status', 'reservation_status_date', 'assigned_room_type', 'booking_changes',
    'adr', 'adr_per_person', 'estimated_booking_value_proxy', 'is_extreme_adr', 'company'
}
assert FORBIDDEN_IN_MODEL.isdisjoint(set(MODEL_FEATURES)), f'Phát hiện cột cấm trong MODEL_FEATURES: {FORBIDDEN_IN_MODEL & set(MODEL_FEATURES)}'

print(f'Số lượng MODEL_FEATURES: {len(MODEL_FEATURES)}')
print('MODEL_FEATURES:', MODEL_FEATURES)

Số lượng MODEL_FEATURES: 41
MODEL_FEATURES: ['hotel', 'lead_time', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'deposit_type', 'agent', 'days_in_waiting_list', 'customer_type', 'required_car_parking_spaces', 'total_of_special_requests', 'arrival_month_num', 'agent_id_present', 'company_id_present', 'total_nights', 'weekend_ratio', 'is_zero_night', 'total_guests', 'has_children_or_babies', 'is_single_adult', 'month_sin', 'month_cos', 'is_peak_season', 'has_previous_cancellations', 'prev_cancel_ratio', 'has_special_requests', 'has_parking', 'is_portugal']


## 4. Sinh Identical Feature Fingerprint (`group_id`) SAU Feature Engineering

Ta sinh `group_id` trực tiếp trên **toàn bộ `MODEL_FEATURES` đã chuẩn hóa** (loại trừ `is_canceled`, `METADATA_COLS` và chính `group_id`) bằng `groupby(..., dropna=False, sort=False).ngroup()`.
- Phương pháp khóa chính xác (exact-key) đảm bảo **0 va chạm băm (zero hash collision)** theo thiết kế.
- Ta kiểm tra lại: phân phối kích thước nhóm, số nhóm có nhãn hỗn hợp (mixed-target groups), và kiểm chứng 1-1 giữa `group_id` và tổ hợp `MODEL_FEATURES`.

In [5]:
# Chuẩn hóa kiểu dữ liệu của MODEL_FEATURES trước khi tạo fingerprint
df_fp = df[MODEL_FEATURES].copy()
for col in df_fp.columns:
    if pd.api.types.is_object_dtype(df_fp[col]) or pd.api.types.is_string_dtype(df_fp[col]):
        df_fp[col] = df_fp[col].astype(str).str.strip()
        df[col] = df_fp[col]
    elif pd.api.types.is_float_dtype(df_fp[col]):
        df_fp[col] = df_fp[col].round(6)

df['group_id'] = df_fp.groupby(MODEL_FEATURES, dropna=False, sort=False).ngroup().astype(np.int64)

# Kiểm chứng zero collision: mỗi group_id chỉ tương ứng đúng 1 tổ hợp duy nhất
n_unique_groups = df['group_id'].nunique()
n_unique_tuples = len(df_fp.drop_duplicates())
assert n_unique_groups == n_unique_tuples, f'Collision detected! {n_unique_groups} != {n_unique_tuples}'

# Kiểm toán phân phối kích thước nhóm và mixed-target groups
group_sizes = df['group_id'].value_counts()
mixed_groups = df.groupby('group_id')[TARGET_COL].nunique()
n_mixed_groups = int((mixed_groups > 1).sum())
n_mixed_rows = int(df['group_id'].isin(mixed_groups[mixed_groups > 1].index).sum())

print(f'Tổng số bản ghi sạch:                    {len(df):,}')
print(f'Số Identical Feature Fingerprint duy nhất: {n_unique_groups:,}')
print(f'Kiểm tra va chạm (Zero Collision Check): ĐẠT ({n_unique_groups:,} == {n_unique_tuples:,})')
print(f'Nhóm có kích thước >= 2:                 {(group_sizes >= 2).sum():,} nhóm ({group_sizes[group_sizes >= 2].sum():,} dòng, {group_sizes[group_sizes >= 2].sum()/len(df)*100:.2f}%)')
print(f'Kích thước nhóm lớn nhất:                {group_sizes.max():,} dòng')
print(f'Nhóm nhãn hỗn hợp (mixed-target):        {n_mixed_groups:,} nhóm ({n_mixed_rows:,} dòng, {n_mixed_rows/len(df)*100:.2f}%)')

# Bảng phân bố kích thước nhóm
size_bins = pd.cut(group_sizes, bins=[0, 1, 2, 5, 10, 50, 10000], labels=['1', '2', '3-5', '6-10', '11-50', '>50'])
display(size_bins.value_counts().sort_index().rename('Số lượng group_id').to_frame())

Tổng số bản ghi sạch:                    119,209
Số Identical Feature Fingerprint duy nhất: 83,536
Kiểm tra va chạm (Zero Collision Check): ĐẠT (83,536 == 83,536)
Nhóm có kích thước >= 2:                 9,089 nhóm (44,762 dòng, 37.55%)
Kích thước nhóm lớn nhất:                180 dòng
Nhóm nhãn hỗn hợp (mixed-target):        382 nhóm (2,921 dòng, 2.45%)


,Số lượng group_id
count,
1,74447
2,5683
3-5,1880
6-10,624
11-50,842
>50,60


## 5. Kiểm tra cuối & Xuất duy nhất `processed/hotel_clean.csv`

Sắp xếp cột rõ ràng (`METADATA_COLS` + `MODEL_FEATURES` + `is_canceled`), kiểm tra không có giá trị `NaN` nào trong `MODEL_FEATURES`, và xuất duy nhất 1 file `processed/hotel_clean.csv`.

In [6]:
ordered_cols = METADATA_COLS + MODEL_FEATURES + [TARGET_COL]
df_out = df[ordered_cols].copy()

assert df_out[MODEL_FEATURES].isna().sum().sum() == 0, 'Vẫn còn NaN trong MODEL_FEATURES!'
assert df_out[TARGET_COL].isna().sum() == 0, 'Target có NaN!'
assert df_out['group_id'].isna().sum() == 0, 'group_id có NaN!'

# Xóa các file cũ nếu có trong processed/ để đảm bảo chỉ có đúng 1 file hotel_clean.csv
for old_f in PROC_DIR.glob('*.csv'):
    if old_f.name != 'hotel_clean.csv':
        old_f.unlink()

df_out.to_csv(OUT_PATH, index=False)
print(f'Đã xuất thành công: {OUT_PATH.resolve()}')
print(f'Kích thước file:    {OUT_PATH.stat().st_size / (1024*1024):.2f} MB')
print(f'Hình dạng dữ liệu:  {df_out.shape[0]:,} dòng x {df_out.shape[1]} cột')
print(f'  - METADATA_COLS ({len(METADATA_COLS)}): {METADATA_COLS}')
print(f'  - MODEL_FEATURES ({len(MODEL_FEATURES)}): {len(MODEL_FEATURES)} cột')
print(f'  - TARGET_COL: {TARGET_COL} (tỷ lệ hủy = {df_out[TARGET_COL].mean()*100:.2f}%)')
display(df_out.head(5))

Đã xuất thành công: E:\repos\ML\MachineLearning\processed\hotel_clean.csv
Kích thước file:    23.90 MB
Hình dạng dữ liệu:  119,209 dòng x 50 cột
  - METADATA_COLS (8): ['row_id', 'group_id', 'arrival_date', 'booking_date', 'arrival_date_year', 'adr', 'estimated_booking_value_proxy', 'is_extreme_adr']
  - MODEL_FEATURES (41): 41 cột
  - TARGET_COL: is_canceled (tỷ lệ hủy = 37.08%)


,row_id,group_id,arrival_date,booking_date,arrival_date_year,adr,estimated_booking_value_proxy,is_extreme_adr,hotel,lead_time,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,...,total_nights,weekend_ratio,is_zero_night,total_guests,has_children_or_babies,is_single_adult,month_sin,month_cos,is_peak_season,has_previous_cancellations,prev_cancel_ratio,has_special_requests,has_parking,is_portugal,is_canceled
0,0,0,2015-07-01,2014-07-24,2015,0.0,0.0,0,Resort Hotel,342,July,27,1,0,0,...,0,0.0,1,2,0,0,-0.5,-0.866025,1,0,0.0,0,0,1,0
1,1,1,2015-07-01,2013-06-24,2015,0.0,0.0,0,Resort Hotel,737,July,27,1,0,0,...,0,0.0,1,2,0,0,-0.5,-0.866025,1,0,0.0,0,0,1,0
2,2,2,2015-07-01,2015-06-24,2015,75.0,75.0,0,Resort Hotel,7,July,27,1,0,1,...,1,0.0,0,1,0,1,-0.5,-0.866025,1,0,0.0,0,0,0,0
3,3,3,2015-07-01,2015-06-18,2015,75.0,75.0,0,Resort Hotel,13,July,27,1,0,1,...,1,0.0,0,1,0,1,-0.5,-0.866025,1,0,0.0,0,0,0,0
4,4,4,2015-07-01,2015-06-17,2015,98.0,196.0,0,Resort Hotel,14,July,27,1,0,2,...,2,0.0,0,2,0,0,-0.5,-0.866025,1,0,0.0,1,0,0,0
